# Advice-marker revision study

This notebook runs the frozen prospective factorial study from a **locally uploaded source bundle**. It does not clone a public branch. The bundle contains the current workspace sources, prompt corpus, frozen Hugging Face revisions, blank human-review template, and detector manifest.

The development path is exploratory and may run before prompt review. Evaluation and longer-budget sensitivity paths stay locked until a human completes the corpus-bound review manifest. Prefix detector windows are sensitivity measurements; only full-response predictions may be compared with full-response human labels.

In [1]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

EXPECTED_BUNDLE_SHA256 = "459984fe0d19dc1df11b7b874d43df011e19194cd765cf2151cd7178df5e3efd"
PINNED_EMBEDDING_MODEL_ID = "intfloat/e5-small-v2"
PINNED_EMBEDDING_REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
PINNED_PREFIX_TOKENIZER_MODEL_ID = "allenai/OLMo-2-0425-1B-Instruct"
PINNED_PREFIX_TOKENIZER_REVISION = "48d788eca847d4d7548f375ad03d3c9312f6139e"
LOCAL_VALIDATION = os.environ.get("REVISION_NOTEBOOK_LOCAL_VALIDATION") == "1"
USE_GOOGLE_DRIVE = False
RUN_DEVELOPMENT = False
RUN_FULL_EVALUATION = False
RUN_LONGER_SENSITIVITY = False
RUN_PREDICTIONS = False
RUN_ANALYSIS = False
TORCH_THREADS = 4
DEVICE = "auto"
DEV_FAMILY_IDS = []  # Empty selects one health and one everyday development family below.

print({"local_validation": LOCAL_VALIDATION, "expected_bundle_sha256": EXPECTED_BUNDLE_SHA256})

{'local_validation': True, 'expected_bundle_sha256': '459984fe0d19dc1df11b7b874d43df011e19194cd765cf2151cd7178df5e3efd'}


## Environment and bundle verification

On Colab, upload `advice_marker_revision_study_bundle.zip` when prompted. Set `USE_GOOGLE_DRIVE=True` only if you want durable run caches in your own Drive.

In [2]:
if not LOCAL_VALIDATION:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.57,<5", "huggingface_hub>=0.34,<2",
        "sentence-transformers>=5,<6", "accelerate>=1.10,<2", "safetensors>=0.6,<1",
    ], check=True)
else:
    print("Local validation: dependency installation skipped.")

Local validation: dependency installation skipped.


In [3]:
bundle_env = os.environ.get("REVISION_NOTEBOOK_BUNDLE")
if bundle_env:
    bundle_path = Path(bundle_env)
else:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Set REVISION_NOTEBOOK_BUNDLE outside Colab.") from exc
    uploaded = files.upload()
    candidates = [Path(name) for name in uploaded if name.endswith(".zip")]
    if len(candidates) != 1:
        raise ValueError("Upload exactly one revision-study bundle zip.")
    bundle_path = candidates[0]

actual_bundle_sha = hashlib.sha256(bundle_path.read_bytes()).hexdigest()
if actual_bundle_sha != EXPECTED_BUNDLE_SHA256:
    raise ValueError(f"Bundle SHA-256 mismatch: {actual_bundle_sha}")

with zipfile.ZipFile(bundle_path) as archive:
    manifest = json.loads(archive.read("bundle_manifest.json").decode("utf-8"))
    workspace_parent = Path(os.environ.get("REVISION_NOTEBOOK_WORKSPACE", "/tmp/advice_marker_revision_bundles"))
    workspace = workspace_parent / manifest["bundle_content_id"]
    workspace.mkdir(parents=True, exist_ok=True)
    for info in archive.infolist():
        member = Path(info.filename)
        if member.is_absolute() or ".." in member.parts:
            raise ValueError(f"Unsafe bundle member: {info.filename}")
    archive.extractall(workspace)

extracted_manifest = json.loads((workspace / "bundle_manifest.json").read_text(encoding="utf-8"))
if extracted_manifest != manifest:
    raise ValueError("Extracted bundle manifest differs from the archive manifest.")
for relative, expected in manifest["files"].items():
    target = workspace / relative
    actual = hashlib.sha256(target.read_bytes()).hexdigest()
    if actual != expected["sha256"] or target.stat().st_size != expected["size_bytes"]:
        raise ValueError(f"Bundled source mismatch: {relative}")
print({"bundle_content_id": manifest["bundle_content_id"], "verified_files": len(manifest["files"])})

{'bundle_content_id': 'bundle_5bf5f0d564120918641c91bcc747b17b', 'verified_files': 12}


In [4]:
if USE_GOOGLE_DRIVE:
    if LOCAL_VALIDATION:
        raise ValueError("Google Drive mounting is disabled during local validation.")
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_ROOT = Path("/content/drive/MyDrive/advice_marker_revision_study")
else:
    CACHE_ROOT = Path(os.environ.get("REVISION_NOTEBOOK_CACHE", "/tmp/advice_marker_revision_cache"))
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(workspace))
from experiments import full_revision_study as full
protocol_path = workspace / "output/revision/full_study/protocol_v2.json"
review_template_path = workspace / "output/revision/full_study/human_prompt_review_v2.json"
protocol = full.load_protocol(protocol_path)
print({"cache_root": str(CACHE_ROOT), "protocol_id": protocol["protocol_id"]})

{'cache_root': 'C:\\Users\\vasilis\\Documents\\GitHub\\llm-research\\output\\revision\\verification\\notebook_dryrun_cache_20260915', 'protocol_id': 'protocol_32f4c8538587e45cdf07e88ed9b1f77d'}


In [5]:
inventory = full.gpu_inventory()
print(json.dumps(inventory, indent=2))
if not LOCAL_VALIDATION and DEVICE in {"auto", "cuda"} and not inventory["cuda_available"]:
    print("No CUDA device is active. Keep full-grid switches off and choose a GPU runtime before generation.")

{
  "python": "3.14.3",
  "platform": "Windows-2025Server-10.0.26100-SP0",
  "torch": "2.13.0+cpu",
  "cuda_available": false,
  "cuda_runtime": null,
  "devices": []
}


## Frozen design and human-review gate

In [6]:
overview = {
    "protocol_status": protocol["status"],
    "evaluation_prompts": 160,
    "checkpoints": len(protocol["checkpoints"]),
    "seeds": protocol["primary_seeds"],
    "primary_expected_responses": protocol["primary_expected_responses"],
    "development_256": protocol["development"]["primary_256_expected_responses"],
    "development_512": protocol["development"]["longer_512_expected_responses"],
    "longer_sensitivity": protocol["longer_budget_sensitivity"]["expected_responses"],
    "human_review_template_status": json.loads(review_template_path.read_text())["status"],
}
print(json.dumps(overview, indent=2))

{
  "protocol_status": "FROZEN_PROTOCOL_PROMPTS_PENDING_HUMAN_MANIPULATION_REVIEW",
  "evaluation_prompts": 160,
  "checkpoints": 6,
  "seeds": [
    0,
    1,
    2
  ],
  "primary_expected_responses": 2880,
  "development_256": 40,
  "development_512": 40,
  "longer_sensitivity": 192,
  "human_review_template_status": "PENDING_HUMAN_REVIEW"
}


In [7]:
# A human reviewer must edit a copy of this file outside this notebook and set every
# family check after manual review. Code never changes PENDING into COMPLETE.
HUMAN_REVIEW_PATH = Path(os.environ.get("REVISION_HUMAN_REVIEW", str(CACHE_ROOT / "human_prompt_review_completed.json")))
if HUMAN_REVIEW_PATH.exists():
    try:
        completed_review = full.validate_human_review(protocol, HUMAN_REVIEW_PATH)
        review_status = completed_review["status"]
    except Exception as exc:
        review_status = f"INVALID: {exc}"
else:
    review_status = "PENDING: copy and manually complete the bundled blank template"
print({"human_review_path": str(HUMAN_REVIEW_PATH), "status": review_status})

{'human_review_path': 'C:\\Users\\vasilis\\Documents\\GitHub\\llm-research\\output\\revision\\verification\\notebook_dryrun_cache_20260915\\human_prompt_review_completed.json', 'status': 'PENDING: copy and manually complete the bundled blank template'}


## Exploratory development run

The default dry run selects two complete four-variant families (one health and one everyday), yielding eight 256-token responses from the pinned 1B Instruct checkpoint. Set `RUN_DEVELOPMENT=True` to execute it.

In [8]:
dev_rows = protocol["development"]["primary_256_grid"]
if not DEV_FAMILY_IDS:
    by_domain = {}
    for row in dev_rows:
        by_domain.setdefault(row["condition"]["domain"], row["scenario_family_id"])
    DEV_FAMILY_IDS = [by_domain["health"], by_domain["everyday"]]
dev_args = [
    "--protocol", str(protocol_path), "--mode", "development-256",
    "--family-ids", *DEV_FAMILY_IDS, "--output-root", str(CACHE_ROOT),
    "--device", DEVICE, "--torch-threads", str(TORCH_THREADS),
]
if RUN_DEVELOPMENT:
    result = full.run_generation(
        protocol_path, "development-256", CACHE_ROOT, DEV_FAMILY_IDS, None,
        DEVICE, False, TORCH_THREADS,
    )
else:
    execution = full.execution_profile(DEVICE, TORCH_THREADS)
    _p, grid, run_dir = full.prepare_run(
        protocol_path, "development-256", CACHE_ROOT, DEV_FAMILY_IDS, None, execution,
    )
    result = {"status": "dry_run", "responses": len(grid), "run_dir": str(run_dir)}
print(json.dumps(result, indent=2))

{
  "status": "dry_run",
  "responses": 8,
  "run_dir": "C:\\Users\\vasilis\\Documents\\GitHub\\llm-research\\output\\revision\\verification\\notebook_dryrun_cache_20260915\\runs\\run_2f0530049d13ba4d2e6ae42e3030b7bb"
}


In [9]:
dev_run_dir = Path(result["run_dir"])
response_files = sorted((dev_run_dir / "responses").glob("*.json")) if (dev_run_dir / "responses").exists() else []
if response_files:
    dev_report = full.verify_run(protocol_path, dev_run_dir, require_complete=False)
    sample = json.loads(response_files[0].read_text())
    print(json.dumps(dev_report, indent=2))
    print({"sample_prompt": sample["generation"]["prompt"], "sample_response": sample["generation"]["response"][:500]})
else:
    print("No development responses yet; this is expected for the default dry run.")

No development responses yet; this is expected for the default dry run.


## Reviewed full grid and longer-budget sensitivity

These cells remain inert until their switches are enabled and `HUMAN_REVIEW_PATH` passes the external manual-review gate. The full grid is always all 2,880 frozen responses; it cannot be subset.

In [10]:
evaluation_result = None
if RUN_FULL_EVALUATION:
    full.validate_human_review(protocol, HUMAN_REVIEW_PATH)
    evaluation_result = full.run_generation(
        protocol_path, "evaluation", CACHE_ROOT, None, HUMAN_REVIEW_PATH,
        DEVICE, False, TORCH_THREADS,
    )
if RUN_LONGER_SENSITIVITY:
    full.validate_human_review(protocol, HUMAN_REVIEW_PATH)
    sensitivity_result = full.run_generation(
        protocol_path, "sensitivity", CACHE_ROOT, None, HUMAN_REVIEW_PATH,
        DEVICE, False, TORCH_THREADS,
    )
print({"evaluation": evaluation_result, "longer_sensitivity_enabled": RUN_LONGER_SENSITIVITY})

{'evaluation': None, 'longer_sensitivity_enabled': False}


In [11]:
exported_responses = None
if evaluation_result and evaluation_result["status"] == "complete":
    evaluation_run_dir = Path(evaluation_result["run_dir"])
    exported_responses = CACHE_ROOT / "exports" / f"{evaluation_run_dir.name}_responses.jsonl"
    export_report = full.export_run(protocol_path, evaluation_run_dir, exported_responses)
    print(json.dumps(export_report, indent=2))
else:
    print("Full-grid export waits for strict verification of all 2,880 responses.")

Full-grid export waits for strict verification of all 2,880 responses.


## Detector predictions and validation sampling

The prediction command emits full-response and 64/128-token rows with pinned dependency identities. Human labels apply only to full responses. The prospective human-validation allocation remains provisional pending development annotation timing.

In [12]:
prediction_output = None
if RUN_PREDICTIONS:
    if exported_responses is None:
        raise ValueError("Complete and export the full evaluation grid first.")
    from huggingface_hub import snapshot_download
    snapshot_download(
        PINNED_EMBEDDING_MODEL_ID, revision=PINNED_EMBEDDING_REVISION,
        allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors"],
    )
    snapshot_download(
        PINNED_PREFIX_TOKENIZER_MODEL_ID, revision=PINNED_PREFIX_TOKENIZER_REVISION,
        allow_patterns=["*.json", "*.txt", "*.model"],
    )
    observed_dependencies = full.shared.prediction_dependencies(include_embedding=True, include_prefix=True)
    expected_dependencies = {
        "embedding_model": (PINNED_EMBEDDING_MODEL_ID, PINNED_EMBEDDING_REVISION),
        "prefix_window_tokenizer": (PINNED_PREFIX_TOKENIZER_MODEL_ID, PINNED_PREFIX_TOKENIZER_REVISION),
    }
    for dependency_name, (model_id, revision) in expected_dependencies.items():
        observed = observed_dependencies[dependency_name]
        if (observed["model_id"], observed["resolved_revision"]) != (model_id, revision):
            raise ValueError(f"Pinned detector dependency mismatch for {dependency_name}: {observed}")
    prediction_output = CACHE_ROOT / "exports" / f"{Path(evaluation_result['run_dir']).name}_union_predictions.jsonl"
    prediction_report = full.predict_complete_evaluation(
        protocol_path, Path(evaluation_result["run_dir"]), prediction_output,
    )
    print(json.dumps(prediction_report, indent=2))
else:
    print("Prediction generation disabled.")

Prediction generation disabled.


In [13]:
validation_plan = {
    "status": "PROVISIONAL_PENDING_DEVELOPMENT_ANNOTATION_TIMING",
    "random_sample": {
        "strata": "6 checkpoints × 2 emotion levels × 2 request types × 2 domains = 48",
        "items_per_stratum": 13,
        "total": 624,
    },
    "positive_enrichment": {
        "maximum_total": 120,
        "illustrative_allocation": "up to 30 per original detector category across the full-response pool",
    },
    "label_scope": "full_response only",
    "prefix_scope": "detector sensitivity only",
    "historical_packet_role": "diagnostic; the 112 historical items are not a held-out prospective set",
}
print(json.dumps(validation_plan, indent=2))

{
  "status": "PROVISIONAL_PENDING_DEVELOPMENT_ANNOTATION_TIMING",
  "random_sample": {
    "strata": "6 checkpoints \u00d7 2 emotion levels \u00d7 2 request types \u00d7 2 domains = 48",
    "items_per_stratum": 13,
    "total": 624
  },
  "positive_enrichment": {
    "maximum_total": 120,
    "illustrative_allocation": "up to 30 per original detector category across the full-response pool"
  },
  "label_scope": "full_response only",
  "prefix_scope": "detector sensitivity only",
  "historical_packet_role": "diagnostic; the 112 historical items are not a held-out prospective set"
}


In [14]:
analysis_script = workspace / "experiments/revision_analysis.py"
if RUN_ANALYSIS:
    if not analysis_script.exists() or prediction_output is None or exported_responses is None:
        raise ValueError("Analysis requires bundled analysis source and complete response/prediction exports.")
    analysis_output = CACHE_ROOT / "checkpoint_analysis"
    subprocess.run([
        sys.executable, "-m", "experiments.revision_analysis",
        "--protocol", str(protocol_path),
        "--responses", str(exported_responses),
        "--predictions", str(prediction_output),
        "--output-dir", str(analysis_output),
        "--bootstrap-replicates", "10000",
        "--bootstrap-seed", "20260915",
        "--confirm-real-complete-archive",
    ], check=True, cwd=workspace)
    print({"analysis_output": str(analysis_output)})
else:
    print("Analysis is disabled until a verified full-grid prediction archive exists.")

Analysis is disabled until a verified full-grid prediction archive exists.


## Run status

A complete scientific run requires: a valid manually completed prompt-review manifest; all 2,880 primary responses under one frozen protocol and execution profile; strict export verification; three detector windows with pinned dependencies; and separately collected prospective human annotations. This notebook does not claim any of those stages completed merely because the workflow is runnable.